# Qwen formal evaluation with four conditions
Run top to bottom. The default checks the saved batch on Drive, without installing model dependencies. Enable RUN_MISSING only to generate absent files with the archived baseline implementation. Existing failed or truncated attempts remain recorded.

In [ ]:
import json
from collections import Counter
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
PROCESSOR_REVISION = MODEL_REVISION
MODEL_TAG = "qwen3-vl-8b"
RESULTS_ROOT = Path("/content/drive/MyDrive/ams-formal-results")
RUN_LABEL = "heldout-evidence-revision-v23-001"
BUDGET = 4
BATCH_NUMBER = 1  # @param {type:"integer"}
RUN_MISSING = False  # @param {type:"boolean"}
BATCHES = [
    [2305, 17398, 1934, 2273, 1664, 4470],
    [2779, 3306, 2274, 1123, 8485, 4805],
    [766, 1061, 3566, 4315, 1907, 2492],
    [3249, 2469, 3029, 2667, 845, 2725],
    [2006, 3120, 3255, 900, 1501, 2798],
]
if BATCH_NUMBER not in range(1, 6):
    raise ValueError("Choose batch 1 through 5.")

In [ ]:
def inspect_batch() -> Counter:
    counts = Counter()
    for target in BATCHES[BATCH_NUMBER - 1]:
        folder = RESULTS_ROOT / RUN_LABEL / MODEL_TAG / f"crater-{target}" / f"budget-{BUDGET}"
        for q in ("Q1", "Q2", "Q3", "Q4"):
            for condition in ("NO_DATA", "ALL_AVAILABLE", "RANDOM", "AGENT"):
                for draw in range(5 if condition == "RANDOM" else 1):
                    path = folder / f"{q}__{condition}__r{draw + 1:02d}.json"
                    if not path.exists():
                        counts["missing"] += 1
                        continue
                    record = json.loads(path.read_bytes())
                    identity = (
                        record["model"],
                        str(record["case_id"]),
                        record["question_id"],
                        record["condition"],
                        record["draw_index"],
                    )
                    if identity != (MODEL_ID, str(target), q, condition, draw):
                        raise ValueError(f"Record identity mismatch: {path}")
                    counts[record["status"]] += 1
    return counts


print("Batch", BATCH_NUMBER, dict(inspect_batch()))

In [ ]:
if RUN_MISSING and inspect_batch()["missing"]:
    import hashlib
    import subprocess

    from IPython import get_ipython

    PROJECT = Path("/content/autonomous-modality-selection")
    commit = "dd429496a0b4cd429818cc789a9a10a3df941805"
    if not PROJECT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/liaojiqichi/autonomous-modality-selection.git",
                str(PROJECT),
            ],
            check=True,
        )
        subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", commit], check=True)
    head = subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip()
    dirty = subprocess.check_output(
        ["git", "-C", str(PROJECT), "status", "--porcelain", "--untracked-files=no"], text=True
    )
    if head != commit or dirty.strip():
        raise RuntimeError("Use a fresh Colab runtime with the pinned project checkout.")
    REVISION_NOTEBOOK = (
        PROJECT / "notebooks/autonomous_modality_selection_answer_revision_formal.ipynb"
    )
    raw = REVISION_NOTEBOOK.read_bytes()
    if (
        hashlib.sha256(raw).hexdigest()
        != "716eeb6ab274ca096143df22bc712b9f1bea871a07b92ea4cfbff631731bc325"
    ):
        raise ValueError("Original notebook fingerprint differs.")
    source_cells = json.loads(raw)["cells"]
    DATASET_ROOT = Path("/content/drive/MyDrive/mercury-heldout-four-modality-v2-20261008")
    DATASET_ZIP = DATASET_ROOT.with_suffix(".zip")
    DATASET_EXTRACT = Path("/content/drive/MyDrive/ams-formal-dataset-extracted")
    REVISION_AUDIT_TOKENS, REVISION_REVIEW_TOKENS = 2048, 512
    shell = get_ipython()
    for index in range(2, 13):
        shell.run_cell("".join(source_cells[index]["source"]), store_history=True).raise_error()
    if shell.user_ns["BATCH_TARGET_IDS"] != BATCHES[BATCH_NUMBER - 1]:
        raise ValueError("Frozen target batch differs.")
    shell.user_ns["SELECTOR_SOURCE"] = r'''import random
import hashlib
import re
from itertools import combinations

from pydantic import BaseModel, ConfigDict, Field


class AgentChoice(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    option_id: str
    rationale: str = Field(min_length=1)


def parse_agent_choice(raw: str) -> AgentChoice:
    """Parse one-shot-json-envelope-v1: bare JSON or one complete JSON fence.

    Original output stays in GENERATION_LOG. No repairs or generation retries occur.
    """
    text = raw.strip()
    if text.startswith("```"):
        match = re.fullmatch(r"```(?:json)?[ \t]*\r?\n([\s\S]*?)\r?\n```", text)
        if match is None or "```" in match.group(1):
            raise ValueError("INVALID_AGENT_JSON_ENVELOPE")
        text = match.group(1).strip()
    return AgentChoice.model_validate_json(text)


def valid_selection(selected: object) -> list[str]:
    """Validate RANDOM/AGENT selections; ALL_AVAILABLE has no selection-budget cap."""
    if not isinstance(selected, list) or not all(isinstance(x, str) for x in selected):
        raise ValueError("Selection must be a list of strings.")
    if len(selected) != 2 or len(set(selected)) != len(selected):
        raise ValueError("Invalid selection count or duplicate modalities.")
    if set(selected) - set(ASSETS):
        raise ValueError("Unknown modality.")
    if sum(ASSETS[x]["cost"] for x in selected) > BUDGET:
        raise ValueError("Selection exceeds the budget.")
    return sorted(selected)


FEASIBLE = [
    list(items)
    for items in combinations(sorted(ASSETS), 2)
    if sum(ASSETS[x]["cost"] for x in items) <= BUDGET
]
OPTIONS = {
    f"S{i + 1}": {"selected": items, "total_cost": sum(ASSETS[x]["cost"] for x in items)}
    for i, items in enumerate(FEASIBLE)
}
if not OPTIONS:
    raise ValueError("NO_FEASIBLE_OPTIONS")


def select_inputs(
    condition: str, question: str, question_id: str, draw_index: int = 0
) -> tuple[list[str], str]:
    """Retain three positional arguments and optional draw_index, without saved aliases."""
    if type(draw_index) is not int or not 0 <= draw_index < RANDOM_DRAWS:
        raise ValueError("Invalid draw_index.")
    if condition != "RANDOM" and draw_index != 0:
        raise ValueError("Non-random conditions have one attempt.")
    if condition == "NO_DATA":
        return [], "No additional scientific evidence."
    if condition == "ALL_AVAILABLE":
        return sorted(ASSETS), "All available modalities; non-budget-matched reference."
    if condition == "RANDOM":
        payload = json.dumps(
            [SEED, f"{CASE_ID}-{question_id}", float(BUDGET), draw_index],
            separators=(",", ":"),
        )
        draw_seed = int.from_bytes(hashlib.sha256(payload.encode()).digest()[:8], "big")
        rng = random.Random(draw_seed)
        chosen = (
            sorted(rng.sample(sorted(ASSETS), 2))
            if BUDGET == 4
            else rng.choice(FEASIBLE)
        )
        return valid_selection(chosen), "Frozen shared random draw."
    if condition != "AGENT":
        raise ValueError(f"Unknown condition: {condition}")

    instruction = {
        "task": SELECTOR_TASK,
        "alignment_version": SELECTOR_ALIGNMENT_VERSION,
        "answer_requirements": next(
            q["answer_requirements"] for q in QUESTIONS if q["question_id"] == question_id
        ),
        "question": question,
        "modality_descriptions": ASSETS,
        "feasible_options": OPTIONS,
        "rules": [
            "Choose exactly two distinct modalities by selecting one supplied option_id; "
            "do not change the pair.",
            "Budget is a ceiling; spending less receives no separate reward.",
            "The catalog is one historical target record, not population data.",
            "Check content_inventory for actual and absent fields; unlisted fields are unknown.",
            "Inputs include plotted views, terrain samples and regional multispectral statistics, "
            "not executable rasters.",
            "Acknowledge representation limitations; "
            "focus on analytical approaches and explanations.",
            "Return only JSON with exactly option_id and a nonempty rationale.",
            "Write the rationale entirely in English.",
            "The rationale should be less than 200 characters.",
        ],
    }
    raw = generate_reply(
        [
            {
                "role": "user",
                "content": [{"type": "text", "text": json.dumps(instruction, ensure_ascii=False)}],
            }
        ],
        max_new_tokens=SELECTOR_TOKENS,
    )
    # GENERATION_LOG retains raw output for both successful and failed attempts.
    if GENERATION_LOG[-1]["finish_reason"] != "eos":
        raise ValueError("SELECTOR_OUTPUT_NOT_COMPLETE")
    parsed = parse_agent_choice(raw)
    if parsed.option_id not in OPTIONS or not parsed.rationale.strip():
        raise ValueError("INVALID_AGENT_CHOICE")
    return valid_selection(OPTIONS[parsed.option_id]["selected"]), parsed.rationale


assert MAX_MODALITIES == 2
assert len(FEASIBLE) == (6 if BUDGET == 4 else 3)
CONFIG["selection_cardinality_version"] = "exact-two-1.0"
assert select_inputs("NO_DATA", "test", "Q1") == ([], "No additional scientific evidence.")
print("Feasible combinations:", OPTIONS)
print(
    "Five random draws for Q1 (repeated combinations are valid):",
    [select_inputs("RANDOM", "test", "Q1", i)[0] for i in range(RANDOM_DRAWS)],
)
'''
    for TARGET_ID in shell.user_ns["BATCH_TARGET_IDS"]:
        print("Running baseline conditions for target", TARGET_ID)
        for source in (
            shell.user_ns["TARGET_SETUP_SOURCE"],
            shell.user_ns["SELECTOR_SOURCE"],
            shell.user_ns["ANSWER_SOURCE"],
            shell.user_ns["EVIDENCE_SOURCE"],
            shell.user_ns["EXPERIMENT_SOURCE"],
        ):
            shell.run_cell(source, store_history=True).raise_error()
        shell.user_ns["GENERATION_LOG"].clear()
        shell.user_ns["gc"].collect()
        shell.user_ns["torch"].cuda.empty_cache()
print("Recorded statuses:", dict(inspect_batch()))